In [1]:
# try reading the gzips

import tqdm
import pandas as pd
import tempfile
import csv
import gzip
import xml.etree.cElementTree as ET
import glob
import random
from readers import pmreader2 as pmreader
import json
import sys
import os

In [2]:
def abstract_plaintext(abstract):
    out = []
    for sec in abstract:
        if sec["header"] != "_UNSTRUCTURED":
            out.append(sec["header"])
        out.append(sec["text"])

    return "\n".join(out)

In [3]:
def iter_abstracts(filename, num_to_select = None):
    with open(filename, 'rb') as f:
        decompressedFile = gzip.GzipFile(fileobj=f, mode='rb')
        xml_dump = decompressedFile.read()
        xml_parsed = ET.fromstring(xml_dump)
        articles = xml_parsed.findall('PubmedArticle/MedlineCitation')
        if num_to_select:
            article_sample = random.sample(articles, num_to_select)
        else:
            article_sample = articles
        for j, article in enumerate(tqdm.tqdm(article_sample)):
            p_article = pmreader.PubmedCorpusReader(xml_ET=article)
            yield p_article.to_dict()


In [4]:
    num_to_select = None
    seed = 42

    train_set_size = 8000000
    val_set_size = 600000
    test_non_sys_size = 600000
    test_sys_size = 300000 

    filenames = glob.glob('/data/iain/pubmed/*.gz')
#     all_data = []


    is_train_data = False
    is_val_data = False
    is_test_sys_data = False
    is_test_non_sys_data = False
    
    
    train_data = pd.DataFrame()
    val_data = pd.DataFrame()
    test_non_sys_data = pd.DataFrame()
    test_sys_data = pd.DataFrame()


    num_todo = len(filenames)
    
    train_per_file = round(train_set_size/num_todo)
    val_per_file = round(val_set_size/num_todo)
    test_non_sys_per_file = round(test_non_sys_size/num_todo)
    test_sys_size_per_file = round(test_sys_size/num_todo)
#     for i, fn in tqdm.tqdm(enumerate(filenames)): # this line does *all* >900 files (takes ages)
    # for id in range(len(filenames)):
    for i, fn in tqdm.tqdm(enumerate([filenames[900]])): # file 900 looks to me to be a decent example; perhaps others better! but some earlier abstracts don't contain much info


            # here we're iterating through *all* files
            # in the final version, probably we'd want to randomly sample, to generate a manageable amount of data
            # I suspect we might not have enough RAM to hold all 300M articles at once, (and so you will need to sample in the loop)
            # but possibly it might work without this, and you could sample from the Pandas dataframe below instead!
            sys.__stdout__.write("file number {}/{} ({}) processing \n".format(i, num_todo, fn)) # optional to see progress from terminal
            print("file number {}/{} ({}) processing \n".format(i, num_todo, fn))
            data = []
            print(fn)
            
            if len(train_data) > train_set_size and len(val_data) > val_set_size and len(test_non_sys_data) > test_non_sys_size and len(test_sys_data) > test_sys_size:
                    break
            
            try:
                for article in iter_abstracts(fn, num_to_select):
                    data.append(article)

                data_pd = pd.DataFrame(data)
                data_pd = data_pd[data_pd['year'] != '']
                data_pd['year'] = list(map(int, data_pd['year']))
                
                
                if len(train_data) < train_set_size:
                    print('entered train')
                    train_subset = data_pd[(data_pd['year']>=1987) & (data_pd['year']<=2014)]
                    if not is_train_data:
                        train_data = train_subset
                        is_train_data = True
                    else:
                        train_data = pd.concat([train_data, train_subset])
                    train_data.to_csv('pubmed_articles_train.csv')
                
                if len(val_data) < val_set_size:
                    print('entered val')
                    val_subset = data_pd[data_pd['year']==2015]
                    if not is_val_data:
                        val_data = val_subset
                        is_val_data = True
                    else:
                        val_data = pd.concat([val_data, val_subset])
                    val_data.to_csv('pubmed_articles_val.csv')
                    
                if len(test_non_sys_data) < test_non_sys_size:
                    print('entered test_non_sys')
                    test_non_sys_subset = data_pd[data_pd['year']==2016]
                    if not is_test_non_sys_data:
                        test_non_sys_data = test_non_sys_subset
                        is_test_non_sys_data = True
                    else:
                        test_non_sys_data = pd.concat([test_non_sys_data, test_non_sys_subset])
                    test_non_sys_data.to_csv('pubmed_articles_test_non_sys.csv')
                    
                if len(test_sys_data) < test_sys_size:
                    print('entered test_sys')
                    test_sys_subset = data_pd[data_pd['year']==2019]
                    if not is_test_sys_data:
                        test_sys_data = test_sys_subset
                        is_test_sys_data = True
                    else:
                        test_sys_data = pd.concat([test_sys_data, test_sys_subset])
                    test_sys_data.to_csv('pubmed_articles_test_sys.csv')
                
            except:
                pass

0it [00:00, ?it/s]

file number 0/1062 (/data/iain/pubmed/pubmed21n0905.xml.gz) processing 

/data/iain/pubmed/pubmed21n0905.xml.gz



 52%|█████▏    | 15503/30000 [00:15<00:14, 1000.50it/s]


100%|██████████| 30000/30000 [00:28<00:00, 1042.10it/s][A


entered train


1it [00:47, 47.78s/it]

entered val
entered test_non_sys
entered test_sys


In [14]:
test_sys_data['indexing_method']

88       Curated
94       Curated
99         Human
986        Human
987        Human
          ...   
29509      Human
29536      Human
29538      Human
29573      Human
29803    Curated
Name: indexing_method, Length: 111, dtype: object

In [32]:
val_subset = data_pd[data_pd['year']==2015]

In [32]:
# num_to_select = None

# filenames = glob.glob('/data/iain/pubmed/*.gz')

# # done_jsons = glob.glob('../robotlabs/trialstreamer/classifications/*.json')
# # done_files = [os.path.join('../robotlabs/trialstreamer/data/', os.path.splitext(os.path.basename(f))[0] + '.gz') for f in done_jsons]

# # files_todo = list(set(filenames) - set(done_files))
# num_todo = len(filenames)

# # all_data = []

# is_article_data = False
# article_data = pd.DataFrame()

# # NOTE to Gregory
# # To run the full version of this will take a long time (since 300 million articles in 900 files)
# # I suggest to run the version below with a single file (which will take a minute or two)
# # to refine the strategy, and then when all is good run it across the full set (by uncommenting and commenting the lines below)


# # for i, fn in tqdm.tqdm(enumerate(filenames[33:])): # this line does *all* >900 files (takes ages)

# # for id in range(len(filenames)):
# for i, fn in tqdm.tqdm(enumerate([filenames[900]])): # file 900 looks to me to be a decent example; perhaps others better! but some earlier abstracts don't contain much info


#         # here we're iterating through *all* files
#         # in the final version, probably we'd want to randomly sample, to generate a manageable amount of data
#         # I suspect we might not have enough RAM to hold all 300M articles at once, (and so you will need to sample in the loop)
#         # but possibly it might work without this, and you could sample from the Pandas dataframe below instead!
#         sys.__stdout__.write("file number {}/{} ({}) processing \n".format(i, num_todo, fn)) # optional to see progress from terminal
#         print("file number {}/{} ({}) processing \n".format(i, num_todo, fn))
#         data = []
#         print(fn)
#         for article in iter_abstracts(fn, num_to_select):
#             data.append(article)
            
#         data_pd = pd.DataFrame(data)
#         data_pd = data_pd[data_pd['year'] != '']
#         data_pd['year'] = list(map(int, data_pd['year']))
#         data_pd = data_pd[data_pd['year']>1987]
        

# #         all_data.extend(data)
#         if not is_article_data:
#             article_data = data_pd
#             is_article_data = True
#         else:
#             article_data = pd.concat([article_data, data_pd])
    
# #     article_data = pd.DataFrame(all_data)
# #     ptyps = []

# #     for ptyprow in article_data['ptyp']:
# #         ptyps.extend(ptyprow)

# #     ptyps = list(set(ptyps))
# #     for ptyp in ptyps:
# #         if 'gov' in ptyp.lower():
# #             print(ptyp)
        



0it [00:00, ?it/s]

file number 0/1062 (/data/iain/pubmed/pubmed21n0905.xml.gz) processing 

/data/iain/pubmed/pubmed21n0905.xml.gz



100%|██████████| 30000/30000 [00:19<00:00, 1573.10it/s]
1it [00:42, 42.73s/it]


In [33]:
len(article_data)

28176

In [24]:
article_data = pd.DataFrame(all_data)

In [25]:
# and here is the final dataset, which can be saved as a CSV
article_data = article_data[article_data['year'] != '']
article_data['year'] = list(map(int, article_data['year']))
article_data_modern = article_data[article_data['year']>1987]

In [28]:
article_data_modern

,title,abstract,authors,journal,journal_abbrv,year,mesh,month,volume,issue,pages,ptyp,pmid,registry_ids,dois
0,"Targeting HDAC3, a new partner protein of AKT ...","[{'header': '_UNSTRUCTURED', 'text': 'Resistan...","[{'Initials': 'J', 'LastName': 'Long', 'ForeNa...",Leukemia,Leukemia,2017,"[Animals, Antineoplastic Agents, Cell Line, Tu...",12,31,12,"{'page_from': '2761', 'page_to': '2770'}","[Journal Article, Research Support, Non-U.S. G...",28462918,[],[10.1038/leu.2017.130]
1,Targeting antigen-independent proliferation in...,"[{'header': '_UNSTRUCTURED', 'text': 'The clin...","[{'Initials': 'E', 'LastName': 'Slinger', 'For...",Leukemia,Leukemia,2017,"[Antigens, Neoplasm, Biomarkers, CD40 Antigens...",12,31,12,"{'page_from': '2601', 'page_to': '2607'}",[Journal Article],28462919,[],[10.1038/leu.2017.129]
2,Evaluating the association of single-nucleotid...,"[{'header': '_UNSTRUCTURED', 'text': 'Higher e...","[{'Initials': 'SM', 'LastName': 'Baxi', 'ForeN...",The pharmacogenomics journal,Pharmacogenomics J,2018,"[ATP Binding Cassette Transporter, Subfamily G...",04,18,2,"{'page_from': '245', 'page_to': '250'}","[Journal Article, Research Support, N.I.H., Ex...",28462920,[],[10.1038/tpj.2017.3]
3,One amino acid makes a difference-Characteriza...,"[{'header': '_UNSTRUCTURED', 'text': 'Thiopuri...","[{'Initials': 'YPH', 'LastName': 'Iu', 'ForeNa...",Scientific reports,Sci Rep,2017,"[Alleles, Genotype, Humans, Methyltransferases...",05,7,,"{'page_from': '46428', 'page_to': '46428'}","[Journal Article, Research Support, Non-U.S. G...",28462921,[],[10.1038/srep46428]
4,Corrigendum: A Comparison of Rule-based Analys...,[],"[{'Initials': 'M', 'LastName': 'Haghighi', 'Fo...",Scientific reports,Sci Rep,2017,[],05,7,,"{'page_from': '39723', 'page_to': '39723'}","[Journal Article, Published Erratum]",28462922,[],[10.1038/srep39723]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,Why we need fairer allocation rules for patien...,[],"[{'Initials': 'HJ', 'LastName': 'Metselaar', '...",Transplant international : official journal of...,Transpl Int,2017,"[Carcinoma, Hepatocellular, Humans, Liver Neop...",11,30,11,"{'page_from': '1092', 'page_to': '1094'}","[Journal Article, Comment]",28493615,[],[10.1111/tri.12980]
29996,Impact of a webcast on nurses' delivery of tob...,"[{'header': 'AIMS AND OBJECTIVES', 'text': 'To...","[{'Initials': 'L', 'LastName': 'Sarna', 'ForeN...",Journal of clinical nursing,J Clin Nurs,2018,"[Adult, Attitude of Health Personnel, Evidence...",Jan,27,1-2,"{'page_from': 'e91', 'page_to': 'e99'}","[Comparative Study, Journal Article]",28493616,[],[10.1111/jocn.13875]
29997,Anti-inflammatory bioactive equivalence of com...,"[{'header': '_UNSTRUCTURED', 'text': 'Bioactiv...","[{'Initials': 'Y', 'LastName': 'Cui', 'ForeNam...",Journal of separation science,J Sep Sci,2017,"[Alkaloids, Anti-Inflammatory Agents, Arenaria...",Jul,40,14,"{'page_from': '2895', 'page_to': '2905'}",[Journal Article],28493617,[],[10.1002/jssc.201700144]
29998,Pondering practice: Enhancing the art of refle...,"[{'header': 'AIMS AND OBJECTIVES', 'text': 'Th...","[{'Initials': 'C', 'LastName': 'Hayes', 'ForeN...",Journal of clinical nursing,J Clin Nurs,2018,"[Clinical Competence, Education, Nursing, Bacc...",Jan,27,1-2,"{'page_from': 'e345', 'page_to': 'e353'}",[Journal Article],28493618,[],[10.1111/jocn.13876]


In [9]:
article_data.describe()

,title,abstract,authors,journal,journal_abbrv,year,mesh,month,volume,issue,pages,ptyp,pmid,registry_ids,dois
count,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000,3000
unique,2980,2591,2972,1554,1553,13,2240,25,275,95,2667,156,3000,39,2895
top,,[],[],PloS one,PLoS One,2017,[],,12,,"{'page_from': '', 'page_to': ''}",[Journal Article],28473154,[],[]
freq,16,410,22,56,56,2498,761,514,110,505,151,1510,1,2962,106


In [15]:
set(article_data['year'])

{'',
 '1976',
 '1981',
 '1986',
 '1991',
 '1993',
 '1997',
 '1998',
 '2000',
 '2001',
 '2002',
 '2003',
 '2004',
 '2005',
 '2008',
 '2010',
 '2011',
 '2012',
 '2013',
 '2014',
 '2015',
 '2016',
 '2017',
 '2018',
 '2019',
 '2020'}

In [15]:
article_data.columns

Index(['title', 'abstract', 'authors', 'journal', 'journal_abbrv', 'year',
       'mesh', 'month', 'volume', 'issue', 'pages', 'ptyp', 'pmid',
       'registry_ids', 'dois'],
      dtype='object')

In [16]:
ptyps = []

for ptyprow in article_data['ptyp']:
    ptyps.extend(ptyprow)
    
ptyps = set(ptyps)

In [17]:
ptyps

{'Address',
 'Autobiography',
 'Biography',
 'Case Reports',
 'Classical Article',
 'Clinical Conference',
 'Clinical Study',
 'Clinical Trial',
 'Clinical Trial, Phase I',
 'Clinical Trial, Phase II',
 'Clinical Trial, Phase III',
 'Clinical Trial, Phase IV',
 'Comment',
 'Comparative Study',
 'Congress',
 'Consensus Development Conference',
 'Consensus Development Conference, NIH',
 'Controlled Clinical Trial',
 'Dataset',
 'Editorial',
 'English Abstract',
 'Equivalence Trial',
 'Evaluation Study',
 'Expression of Concern',
 'Festschrift',
 'Guideline',
 'Historical Article',
 'Interview',
 'Introductory Journal Article',
 'Journal Article',
 'Lecture',
 'Letter',
 'Meta-Analysis',
 'Multicenter Study',
 'News',
 'Observational Study',
 'Patient Education Handout',
 'Personal Narrative',
 'Portrait',
 'Practice Guideline',
 'Pragmatic Clinical Trial',
 'Published Erratum',
 'Randomized Controlled Trial',
 'Research Support, N.I.H., Extramural',
 'Research Support, N.I.H., Intramural

In [18]:
proposed_design_types = [
    
    # Propose to compress these into single category
    'Editorial',
    'Address',
    'Letter',
    'Lecture',
    #######################################
    'Adaptive Clinical Trial',
    
     
    
    # Propose to compress these into single category
    'Clinical Trial',
    'Clinical Trial, Phase I', 
    'Clinical Trial, Phase II', 
    'Clinical Trial, Phase III',
    'Clinical Trial, Phase IV',
    #######################################
    'Clinical Trial Protocol',
    
    # Propose to compress these into single category
    'Consensus Development Conference',
    'Consensus Development Conference, NIH',
    ########################################
    
    # Propose to compress these into single category
    'Guideline',
    'Practice Guideline',
    ########################################
    
    'Case Reports',
    'Controlled Clinical Trial',
    'Observational Study',
    'Randomized Controlled Trial',
    'Comment',
    'Comparative Study',
    'Equivalence Trial',
    'Evaluation Study',
    'Government Publications',
    'Randomized Controlled Trial',
    'Government Publication',
    'Meta-Analysis',
    'Personal Narrative',
    'Pragmatic Clinical Trial',
    'Review',
    'Systematic Review',
    'Validation Study'

]


In [33]:
article_data.to_csv('pubmed_articles.csv')

In [37]:
proposed_design_types

['Editorial',
 'Address',
 'Letter',
 'Lecture',
 'Adaptive Clinical Trial',
 'Clinical Trial',
 'Clinical Trial, Phase I',
 'Clinical Trial, Phase II',
 'Clinical Trial, Phase III',
 'Clinical Trial, Phase IV',
 'Clinical Trial Protocol',
 'Consensus Development Conference',
 'Consensus Development Conference, NIH',
 'Guideline',
 'Practice Guideline',
 'Case Reports',
 'Controlled Clinical Trial',
 'Observational Study',
 'Randomized Controlled Trial',
 'Comment',
 'Comparative Study',
 'Equivalence Trial',
 'Evaluation Study',
 'Government Publications',
 'Randomized Controlled Trial',
 'Government Publication',
 'Meta-Analysis',
 'Personal Narrative',
 'Pragmatic Clinical Trial',
 'Review',
 'Systematic Review',
 'Validation Study']

In [41]:
[ set(ptyps).intersection(set(proposed_design_types)) ==set() for ptyps in article_data['ptyp']]

[True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 False,
 True,
 True,
 True,
 True,
 True,
 True,
 False,
 False,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 False,
 True,
 False,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 False,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 False,
 True,
 True,
 True,
 False,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 False,
 True,
 False,
 False,
 True,
 False,
 True,
 True,
 True,
 False,
 True,
 False,
 True,
 False,
 True,
 True,
 True,
 True,
 False,
 True,
 False,
 True,
 True,
 False,
 True,
 True,
 True,
 True,
 True,
 True,
 False,
 True,
 True,
 False,
 False,
 True,
 True,
 True,
 True,
 False,
 True,
 True,
 True,
 True,
 False,
 True,
 True,
 True,
 True,
 True,
 True,
 False,
 True,
 True,
 True,
 True,
 True,
 Tr

In [43]:
pos_articles = article_data[[ set(ptyps).intersection(set(proposed_design_types)) ==set() for ptyps in article_data['ptyp']] ]

In [44]:
neg_articles = article_data[[ set(ptyps).intersection(set(proposed_design_types)) !=set() for ptyps in article_data['ptyp']] ]

In [45]:
len(pos_articles)

20298

In [46]:
len(neg_articles)

9702

In [49]:
len(article_data)

30000

In [13]:
# num_types_per_art = [0]
# for i in range(len(article_data)):
#     art = article_data.loc[i]
#     for art in art['ptyp']:
#         if art in proposed_design_types:
#             num_types_per_art[-1]+=1
#     num_types_per_art.append(0)

In [14]:
# clashes = [(article_data.iloc[i], num_types_per_art[i]) for i in range(len(num_types_per_art)) if num_types_per_art[i]>1]
# print(clashes)
# print(len(clashes))


[(title            European evidence-based recommendations for di...
abstract         [{'header': '_UNSTRUCTURED', 'text': 'Antiphos...
authors          [{'Initials': 'N', 'LastName': 'Groot', 'ForeN...
journal                           Annals of the rheumatic diseases
journal_abbrv                                        Ann Rheum Dis
year                                                          2017
mesh             [Adolescent, Antiphospholipid Syndrome, Child,...
month                                                          Oct
volume                                                          76
issue                                                           10
pages                     {'page_from': '1637', 'page_to': '1641'}
ptyp             [Journal Article, Practice Guideline, Review, ...
pmid                                                      28473426
registry_ids                                                    []
dois                             [10.1136/annrheumdis-2016-2

In [15]:

# for i in range(len(article_data)):
#     if 'Observational Study' in article_data.iloc[i]['ptyp'] and 'Randomized Controlled Trial' in article_data.iloc[i]['ptyp']:
#         clash = article_data.iloc[i]
#         break

In [16]:
# clash['title']

'Polyphenol Levels Are Inversely Correlated with Body Weight and Obesity in an Elderly Population after 5 Years of Follow Up (The Randomised PREDIMED Study).'

In [18]:
# clash['abstract']

[{'header': '_UNSTRUCTURED',
  'text': 'Overweight and obesity have been steadily increasing in recent years and currently represent a serious threat to public health. Few human studies have investigated the relationship between polyphenol intake and body weight. Our aim was to assess the relationship between urinary polyphenol levels and body weight. A cross-sectional study was performed with 573 participants from the PREDIMED (Prevención con Dieta Mediterránea) trial (ISRCTN35739639). Total polyphenol levels were measured by a reliable biomarker, total urinary polyphenol excretion (TPE), determined by the Folin-Ciocalteu method in urine samples. Participants were categorized into five groups according to their TPE at the fifth year. Multiple linear regression models were used to assess the relationships between TPE and obesity parameters; body weight (BW), body mass index (BMI), waist circumference (WC), and waist-to-height ratio (WHtR). After a five years follow up, significant inve

<function pandas.core.reshape.concat.concat(objs: Union[Iterable[ForwardRef('NDFrame')], Mapping[Union[Hashable, NoneType], ForwardRef('NDFrame')]], axis=0, join='outer', ignore_index: bool = False, keys=None, levels=None, names=None, verify_integrity: bool = False, sort: bool = False, copy: bool = True) -> Union[ForwardRef('DataFrame'), ForwardRef('Series')]>